# Thayer-Net: Controlled Galaxy Deblending

This notebook runs the controlled synthetic deblending evaluation on Galaxy10 DECaLS cutouts. It splits original galaxies before blending, generates foreground-only synthetic blends, trains a compact direct-reconstruction U-Net, and evaluates reconstruction quality with both whole-image and affected-region metrics.


## 1. Setup and Imports


In [ ]:
from collections import defaultdict
from pathlib import Path
import random
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import baselines as gd_baselines
from src import blend as gd_blend
from src import data as gd_data
from src import models as gd_models
from src import train as gd_train
from src import utils as gd_utils

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
rng = np.random.default_rng(seed)

device = gd_train.resolve_device("auto")
print(f"Using device: {device}")


## 2. Configuration

The project config provides portable defaults. This notebook uses the larger evaluation-scale settings associated with the current run, so the effective values are shown explicitly.


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
DATASET_PATH = PROJECT_ROOT / "data" / "Galaxy10_DECals.h5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    default_config = yaml.safe_load(f)

blend_defaults = default_config["blending"]
model_defaults = default_config["model"]
training_defaults = default_config["training"]

config = {
    "dataset_path": str(DATASET_PATH),
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "train_subset": 5000,
    "val_subset": 800,
    "test_subset": 800,
    "n_train_blends": 5000,
    "n_val_blends": 800,
    "n_test_blends": 800,
    "blend_params": {
        "max_shift": blend_defaults["max_shift"],
        "brightness_range": tuple(blend_defaults["brightness_range"]),
        "blur_range": tuple(blend_defaults["blur_range"]),
        "noise_range": tuple(blend_defaults["noise_range"]),
        "rotation_range": tuple(blend_defaults["rotation_range"]),
    },
    "model_params": {
        "in_channels": model_defaults["in_channels"],
        "out_channels": model_defaults["out_channels"],
        "base_channels": model_defaults["base_channels"],
        "norm": model_defaults["norm"],
    },
    "training_params": {
        "batch_size": training_defaults["batch_size"],
        "num_epochs": 20,
        "learning_rate": training_defaults["learning_rate"],
        "weight_decay": training_defaults["weight_decay"],
        "checkpoint_path": str(OUTPUT_DIR / "checkpoints" / "unet_5000_800_20ep.pth"),
    },
    "affected_region_threshold": 0.02,
}

OUTPUT_DIR.mkdir(exist_ok=True)

key_settings = pd.DataFrame(
    [
        {"setting": "dataset_path", "value": str(DATASET_PATH.relative_to(PROJECT_ROOT))},
        {"setting": "train/val/test blends", "value": "5000 / 800 / 800"},
        {"setting": "epochs", "value": config["training_params"]["num_epochs"]},
        {"setting": "max_shift", "value": config["blend_params"]["max_shift"]},
        {"setting": "rotation_range", "value": config["blend_params"]["rotation_range"]},
        {"setting": "affected_region_threshold", "value": config["affected_region_threshold"]},
    ]
)
key_settings


## 3. Load Galaxy10 DECaLS Data

Galaxy10 DECaLS is kept outside version control. Place `Galaxy10_DECals.h5` under `data/` before running the notebook.


In [ ]:
images_raw, labels, metadata = gd_data.load_galaxy10(config["dataset_path"])
images = gd_data.normalise_images(images_raw)

print(f"Loaded images: {images.shape}")
print(f"Labels: {labels.shape}")
print(f"Metadata fields: {sorted(metadata.keys())}")


## 4. Train/Validation/Test Split

Original galaxy images are split before synthetic blending so that target and contaminant cutouts from the same source pool do not leak across train, validation, and test sets.


In [ ]:
# Split original cutouts before blending to avoid synthetic leakage across sets.
(X_train, y_train), (X_val, y_val), (X_test, y_test) = gd_data.split_dataset(
    images,
    labels,
    train_frac=config["train_frac"],
    val_frac=config["val_frac"],
    test_frac=config["test_frac"],
    seed=seed,
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


## 5. Synthetic Blend Generation

Blends are generated by extracting the contaminant foreground, preserving faint halos with a soft central mask, and adding only that foreground layer to the target. This avoids rectangular cutout artifacts that would make the task less astrophysically meaningful.

If `src/blend.py` is edited during an active Jupyter session, restart the kernel or explicitly reload the module and regenerate `blends_train`, `blends_val`, and `blends_test`. Existing blend objects remain in memory and will not update automatically.


In [ ]:
X_train_small = X_train[: config["train_subset"]]
X_val_small = X_val[: config["val_subset"]]
X_test_small = X_test[: config["test_subset"]]

blend_params = config["blend_params"]

# Blend generation adds only extracted contaminant foreground, avoiding full-cutout edges.
blends_train = gd_blend.generate_blends(
    X_train_small,
    n_blends=config["n_train_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)
blends_val = gd_blend.generate_blends(
    X_val_small,
    n_blends=config["n_val_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)
blends_test = gd_blend.generate_blends(
    X_test_small,
    n_blends=config["n_test_blends"],
    max_shift=blend_params["max_shift"],
    brightness_range=blend_params["brightness_range"],
    blur_range=blend_params["blur_range"],
    noise_range=blend_params["noise_range"],
    rotation_range=blend_params["rotation_range"],
    rng=rng,
)

print(f"Train blends: {len(blends_train)}")
print(f"Val blends: {len(blends_val)}")
print(f"Test blends: {len(blends_test)}")


## 6. Visual Check of Synthetic Blends


In [ ]:
sample_indices = rng.choice(len(blends_train), size=min(3, len(blends_train)), replace=False)
fig, axes = plt.subplots(len(sample_indices), 3, figsize=(9, 3 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, idx in enumerate(sample_indices):
    sample = blends_train[int(idx)]
    axes[row, 0].imshow(sample["target"])
    axes[row, 0].set_title("Target")
    axes[row, 1].imshow(sample["contaminant"])
    axes[row, 1].set_title("Contaminant")
    axes[row, 2].imshow(sample["blended"])
    legacy_label = sample["info"].get("generation_difficulty", sample["info"].get("difficulty"))
    axes[row, 2].set_title(f"Blend legacy gen: {legacy_label}")
    for col in range(3):
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()


## 7. Baselines

The identity baseline returns the blended image unchanged. It is deliberately simple: if the learned model cannot beat this baseline where the contaminant changes the image, the deblending setup is not yet useful.


In [ ]:
test_targets = [sample["target"] for sample in blends_test]
test_blended = [sample["blended"] for sample in blends_test]
# Identity leaves the blend unchanged; it is the minimum useful comparison.
identity_outputs = [gd_baselines.identity_baseline(sample["blended"]) for sample in blends_test]
threshold_outputs = [gd_baselines.threshold_baseline(sample["blended"]) for sample in blends_test]

print(f"Prepared {len(identity_outputs)} identity and threshold-baseline predictions.")


## 8. Direct U-Net Model

The direct model maps a blended RGB image to a reconstructed target RGB image.


In [ ]:
train_dataset = gd_train.BlendDataset(blends_train)
val_dataset = gd_train.BlendDataset(blends_val)
test_dataset = gd_train.BlendDataset(blends_test)

model = gd_models.UNet(**config["model_params"]).to(device)
model


## 9. Training


In [ ]:
train_losses, val_losses = gd_train.train_model(
    model,
    train_dataset,
    val_dataset,
    num_epochs=config["training_params"]["num_epochs"],
    batch_size=config["training_params"]["batch_size"],
    learning_rate=config["training_params"]["learning_rate"],
    weight_decay=config["training_params"]["weight_decay"],
    device=device,
    checkpoint_path=config["training_params"]["checkpoint_path"],
)

plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Whole-Image Evaluation

Whole-image metrics measure global reconstruction fidelity. They are useful but can overstate baseline performance because most pixels in a synthetic blend are unchanged relative to the target.


In [ ]:
model_whole_metrics, reconstructions = gd_train.evaluate_model(
    model,
    test_dataset,
    device=device,
)

baseline_whole_metrics = {
    "identity baseline": gd_utils.compute_metrics(identity_outputs, test_targets),
    "threshold baseline": gd_utils.compute_metrics(threshold_outputs, test_targets),
}
whole_image_metrics = {
    **baseline_whole_metrics,
    "model": model_whole_metrics,
}

print("Whole-image metrics")
pd.DataFrame(whole_image_metrics).T


## 11. Affected-Region Evaluation

Affected-region metrics evaluate only pixels where the blend differs from the target. This better isolates deblending performance from the unchanged background and target pixels that make the identity baseline look strong.


In [ ]:
# Restrict these metrics to pixels where blending measurably changed the target.
affected_region_metrics = {
    "identity baseline": gd_utils.compute_affected_region_metrics(
        identity_outputs,
        test_targets,
        test_blended,
        threshold=config["affected_region_threshold"],
    ),
    "threshold baseline": gd_utils.compute_affected_region_metrics(
        threshold_outputs,
        test_targets,
        test_blended,
        threshold=config["affected_region_threshold"],
    ),
    "model": gd_utils.compute_affected_region_metrics(
        reconstructions,
        test_targets,
        test_blended,
        threshold=config["affected_region_threshold"],
    ),
}

print("Affected-region metrics")
pd.DataFrame(affected_region_metrics).T


## 12. Legacy Generation Metadata Analysis

The blend generator records legacy easy/medium/hard labels from controllable generation parameters such as shift, brightness, blur, noise, and size ratio. These labels are useful provenance metadata, but they are not the same as measured blend severity, core obstruction, or model failure.


In [ ]:
def evaluate_by_group(samples, predictions, group_key="generation_difficulty", threshold=0.02):
    rows = []
    groups = defaultdict(list)
    for i, sample in enumerate(samples):
        info = sample["info"]
        group = info.get(group_key, info.get("difficulty"))
        groups[group].append(i)

    for group, indices in groups.items():
        group_targets = [samples[i]["target"] for i in indices]
        group_blends = [samples[i]["blended"] for i in indices]
        group_preds = [predictions[i] for i in indices]

        identity_whole = gd_utils.compute_metrics(group_blends, group_targets)
        model_whole = gd_utils.compute_metrics(group_preds, group_targets)
        identity_masked = gd_utils.compute_affected_region_metrics(
            predictions=group_blends,
            targets=group_targets,
            blends=group_blends,
            threshold=threshold,
        )
        model_masked = gd_utils.compute_affected_region_metrics(
            predictions=group_preds,
            targets=group_targets,
            blends=group_blends,
            threshold=threshold,
        )

        rows.append(
            {
                "group": group,
                "n": len(indices),
                "identity_mse": identity_whole["mse"],
                "model_mse": model_whole["mse"],
                "identity_ssim": identity_whole["ssim"],
                "model_ssim": model_whole["ssim"],
                "identity_masked_mse": identity_masked["masked_mse"],
                "model_masked_mse": model_masked["masked_mse"],
                "identity_masked_mae": identity_masked["masked_mae"],
                "model_masked_mae": model_masked["masked_mae"],
                "mask_fraction": model_masked["mean_mask_fraction"],
            }
        )

    return pd.DataFrame(rows).sort_values("group")


difficulty_results = evaluate_by_group(
    samples=blends_test,
    predictions=reconstructions,
    group_key="generation_difficulty",
    threshold=config["affected_region_threshold"],
)

difficulty_results


## 13. Blend Severity Analysis

Blend severity uses the actual changed pixels in each synthetic blend. It measures contaminant damage to the target, not model difficulty; model failure is tracked separately with model affected error and model-improvement ratios.


In [ ]:
def add_bins_and_evaluate(samples, predictions, threshold=0.02):
    rows = []

    for i, sample in enumerate(samples):
        info = sample["info"]
        target = sample["target"]
        blended = sample["blended"]
        pred = predictions[i]

        identity_masked = gd_utils.compute_affected_region_metrics(
            predictions=[blended],
            targets=[target],
            blends=[blended],
            threshold=threshold,
        )
        model_masked = gd_utils.compute_affected_region_metrics(
            predictions=[pred],
            targets=[target],
            blends=[blended],
            threshold=threshold,
        )

        rows.append(
            {
                "generation_difficulty": info.get("generation_difficulty", info.get("difficulty")),
                "brightness": info["brightness"],
                "size_ratio": info["size_ratio"],
                "shift_distance": abs(info["shift"][0]) + abs(info["shift"][1]),
                "identity_masked_mse": identity_masked["masked_mse"],
                "model_masked_mse": model_masked["masked_mse"],
                "identity_masked_mae": identity_masked["masked_mae"],
                "model_masked_mae": model_masked["masked_mae"],
                "model_improvement_ratio": (
                    identity_masked["masked_mse"] / model_masked["masked_mse"]
                    if model_masked["masked_mse"] > 0
                    else np.nan
                ),
                "model_failure_score": model_masked["masked_mse"],
                "model_delta_mse": model_masked["masked_mse"] - identity_masked["masked_mse"],
            }
        )

    df = pd.DataFrame(rows)
    df["brightness_bin"] = pd.cut(
        df["brightness"],
        bins=[0, 0.7, 0.9, 1.2, np.inf],
        labels=["low", "medium", "high", "very_high"],
    )
    df["size_ratio_bin"] = pd.cut(
        df["size_ratio"],
        bins=[0, 0.5, 1.0, 2.0, np.inf],
        labels=["small_contaminant", "similar_smaller", "similar_larger", "large_contaminant"],
    )
    df["shift_bin"] = pd.cut(
        df["shift_distance"],
        bins=[0, 24, 56, 112, np.inf],
        labels=["high_overlap", "medium_overlap", "low_overlap", "very_low_overlap"],
    )
    return df


per_sample_results = add_bins_and_evaluate(
    samples=blends_test,
    predictions=reconstructions,
    threshold=config["affected_region_threshold"],
)

per_sample_results.head()


In [ ]:
def target_core_mask(target, aperture_fraction=0.18, core_percentile=85.0):
    gray = target.mean(axis=-1)
    height, width = gray.shape
    center_y, center_x = height / 2.0, width / 2.0
    y_grid, x_grid = np.ogrid[:height, :width]
    radius = aperture_fraction * min(height, width)
    aperture = np.hypot(y_grid - center_y, x_grid - center_x) <= radius
    threshold = float(np.percentile(gray[aperture], core_percentile))
    core = aperture & (gray >= threshold)
    return core if np.any(core) else aperture


def compute_blend_severity_table(samples, predictions, threshold=0.02):
    rows = []

    for i, sample in enumerate(samples):
        target = sample["target"]
        blended = sample["blended"]
        pred = predictions[i]
        info = sample["info"]

        affected = gd_utils.affected_region_mask(
            blended=blended,
            target=target,
            threshold=threshold,
        )
        mask_fraction = affected.mean()
        core = target_core_mask(target)
        core_obstruction_fraction = (
            np.logical_and(affected, core).sum() / core.sum() if np.any(core) else 0.0
        )

        diff = blended - target
        affected_mse = np.mean((diff[affected]) ** 2) if affected.any() else 0.0
        affected_mae = np.mean(np.abs(diff[affected])) if affected.any() else 0.0

        model_diff = pred - target
        model_affected_mse = np.mean((model_diff[affected]) ** 2) if affected.any() else 0.0
        model_affected_mae = np.mean(np.abs(model_diff[affected])) if affected.any() else 0.0
        model_improvement_ratio = (
            affected_mse / model_affected_mse if model_affected_mse > 0 else np.nan
        )

        # Blend severity measures contaminant damage, not model difficulty.
        blend_severity_score = mask_fraction * affected_mae * (1.0 + core_obstruction_fraction)
        shift = info.get("shift", (0, 0))

        rows.append(
            {
                "index": i,
                "generation_difficulty": info.get("generation_difficulty", info.get("difficulty")),
                "brightness": info.get("brightness"),
                "size_ratio": info.get("size_ratio"),
                "shift_x": shift[0],
                "shift_y": shift[1],
                "shift_distance": abs(shift[0]) + abs(shift[1]),
                "mask_fraction": mask_fraction,
                "core_obstruction_fraction": core_obstruction_fraction,
                "identity_affected_mse": affected_mse,
                "identity_affected_mae": affected_mae,
                "model_affected_mse": model_affected_mse,
                "model_affected_mae": model_affected_mae,
                "blend_severity_score": blend_severity_score,
                "model_improvement_ratio": model_improvement_ratio,
                "model_failure_score": model_affected_mse,
                "model_delta_mse": model_affected_mse - affected_mse,
            }
        )

    df = pd.DataFrame(rows)
    df["blend_severity_bin"] = pd.qcut(
        df["blend_severity_score"].rank(method="first"),
        q=3,
        labels=["easy", "medium", "hard"],
    )
    df["core_overlap_bin"] = pd.cut(
        df["core_obstruction_fraction"],
        bins=[-0.001, 1.0 / 3.0, 2.0 / 3.0, 1.001],
        labels=["low", "medium", "high"],
        include_lowest=True,
    )
    return df


severity_results = compute_blend_severity_table(
    samples=blends_test,
    predictions=reconstructions,
    threshold=config["affected_region_threshold"],
)

severity_crosstab = pd.crosstab(
    severity_results["generation_difficulty"],
    severity_results["blend_severity_bin"],
    margins=True,
)
blend_severity_summary = severity_results.groupby("blend_severity_bin", observed=False)[[
    "identity_affected_mse",
    "model_affected_mse",
    "identity_affected_mae",
    "model_affected_mae",
    "mask_fraction",
    "core_obstruction_fraction",
    "blend_severity_score",
    "model_improvement_ratio",
    "model_failure_score",
    "model_delta_mse",
]].mean()

display(severity_crosstab)
blend_severity_summary


In [ ]:
best_examples = severity_results.sort_values("model_improvement_ratio", ascending=False).head(5)
worst_examples = severity_results.sort_values("model_failure_score", ascending=False).head(5)

example_rankings = {
    "best_examples": best_examples[[
        "index",
        "generation_difficulty",
        "blend_severity_bin",
        "identity_affected_mse",
        "model_affected_mse",
        "model_improvement_ratio",
        "blend_severity_score",
        "model_failure_score",
        "model_delta_mse",
    ]],
    "worst_examples": worst_examples[[
        "index",
        "generation_difficulty",
        "blend_severity_bin",
        "identity_affected_mse",
        "model_affected_mse",
        "model_improvement_ratio",
        "blend_severity_score",
        "model_failure_score",
        "model_delta_mse",
    ]],
}

display(example_rankings["best_examples"])
example_rankings["worst_examples"]


## 14. Success and Failure Examples

The selected examples are qualitative checks, not a replacement for aggregate metrics.


In [ ]:
def show_reconstruction_example(samples, predictions, index, caption=None, severity_df=None):
    sample = samples[index]
    severity_label = None
    ratio_note = None
    if severity_df is not None:
        row = severity_df.loc[severity_df["index"] == index]
        if len(row) > 0:
            row = row.iloc[0]
            severity_label = row["blend_severity_bin"]
            if "model_improvement_ratio" in row:
                ratio = row["model_improvement_ratio"]
                ratio_note = f"model improvement ratio={ratio:.2f}x"
                if ratio < 1:
                    ratio_note += "; direct U-Net worse than identity"

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))

    images = [
        sample["target"],
        sample["contaminant"],
        sample["blended"],
        predictions[index],
    ]
    blend_title = f"Blend legacy gen: {sample['info'].get('generation_difficulty', sample['info'].get('difficulty'))}"
    if severity_label is not None:
        blend_title += f" / severity={severity_label}"

    titles = [
        "Target",
        "Contaminant",
        blend_title,
        "Reconstruction",
    ]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title)
        ax.axis("off")

    display_caption = caption
    if ratio_note is not None:
        display_caption = f"{caption} ({ratio_note})" if caption else ratio_note
    if display_caption is not None:
        fig.suptitle(display_caption, y=1.03)
    plt.tight_layout()
    plt.show()


In [ ]:
success_index = int(best_examples.iloc[min(4, len(best_examples) - 1)]["index"])
partial_failure_index = int(worst_examples.iloc[0]["index"])
hard_failure_index = int(worst_examples.iloc[min(1, len(worst_examples) - 1)]["index"])

selected_examples = [
    (
        success_index,
        "Example successful reconstruction: contaminant suppressed while target structure is preserved.",
    ),
    (
        partial_failure_index,
        "Partial failure case: model affected error remains high; inspect model-improvement ratio for whether it beats identity.",
    ),
    (
        hard_failure_index,
        "Largest model-failure case: if model-improvement ratio is below 1, the direct U-Net is worse than identity for this example.",
    ),
]

for index, caption in selected_examples:
    show_reconstruction_example(
        blends_test,
        reconstructions,
        index,
        caption=caption,
        severity_df=severity_results,
    )


Model checkpoints and generated artifacts are saved locally under the git-ignored `outputs/` directory; the `.pth` file lives in `outputs/checkpoints/`.

## 15. Save Model Checkpoint, Tables, and Figures

Only reviewed, public-safe figures should be copied into `reports/figures/`.


In [ ]:
# Local artifacts stay under git-ignored outputs/ paths.
checkpoint_dir = OUTPUT_DIR / "checkpoints"
results_dir = OUTPUT_DIR / "results"
figure_dir = OUTPUT_DIR / "figures"
for directory in (checkpoint_dir, results_dir, figure_dir):
    directory.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "unet_direct_5000train_800val_800test_20ep.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": config,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "final_val_loss": val_losses[-1] if val_losses else None,
    },
    checkpoint_path,
)

difficulty_results.to_csv(results_dir / "difficulty_results_direct_unet.csv", index=False)
per_sample_results.to_csv(results_dir / "per_sample_results_direct_unet.csv", index=False)
severity_results.to_csv(results_dir / "measured_severity_results.csv", index=False)
severity_crosstab.to_csv(results_dir / "difficulty_label_crosstab.csv")

saved_tables = sorted(path.name for path in results_dir.glob("*.csv"))
saved_tables


In [ ]:
def save_reconstruction_example(samples, predictions, index, filename, severity_df=None):
    sample = samples[index]
    severity_label = None
    ratio_note = None
    if severity_df is not None:
        row = severity_df.loc[severity_df["index"] == index]
        if len(row) > 0:
            row = row.iloc[0]
            severity_label = row["blend_severity_bin"]
            if "model_improvement_ratio" in row:
                ratio = row["model_improvement_ratio"]
                ratio_note = f"model improvement ratio={ratio:.2f}x"
                if ratio < 1:
                    ratio_note += "; direct U-Net worse than identity"

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    images = [
        sample["target"],
        sample["contaminant"],
        sample["blended"],
        predictions[index],
    ]

    original_label = sample["info"].get("generation_difficulty", sample["info"].get("difficulty"))
    blend_title = f"Blend legacy gen: {original_label}"
    if severity_label is not None:
        blend_title += f" / severity={severity_label}"

    titles = ["Target", "Contaminant", blend_title, "Reconstruction"]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title)
        ax.axis("off")

    if ratio_note is not None:
        fig.suptitle(ratio_note, y=1.03)
    plt.tight_layout()
    fig.savefig(figure_dir / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


figure_exports = {
    "direct_unet_success_example.png": success_index,
    "direct_unet_partial_failure_example.png": partial_failure_index,
    "direct_unet_hard_failure_example.png": hard_failure_index,
}
for filename, index in figure_exports.items():
    save_reconstruction_example(
        blends_test,
        reconstructions,
        index,
        filename,
        severity_df=severity_results,
    )

identity_whole = whole_image_metrics["identity baseline"]
model_whole = whole_image_metrics["model"]
identity_region = affected_region_metrics["identity baseline"]
model_region = affected_region_metrics["model"]
affected_mse_ratio = identity_region["masked_mse"] / model_region["masked_mse"]
affected_region_improvement = pd.DataFrame(
    [
        {
            "identity_masked_mse": identity_region["masked_mse"],
            "model_masked_mse": model_region["masked_mse"],
            "model_improvement_ratio": affected_mse_ratio,
        }
    ]
)
display(affected_region_improvement)

log_path = OUTPUT_DIR / "experiment_log_direct_unet.md"
log_text = f"""
# Direct U-Net Evaluation

Training setup:
- Train blends: {config["n_train_blends"]:,}
- Validation blends: {config["n_val_blends"]:,}
- Test blends: {config["n_test_blends"]:,}
- Epochs: {config["training_params"]["num_epochs"]}
- Model: compact U-Net
- Task: blended RGB image -> clean target RGB image

Main result:
- Whole-image MSE improved from {identity_whole["mse"]:.6f} to {model_whole["mse"]:.6f}.
- Affected-region MSE improved from {identity_region["masked_mse"]:.6f} to {model_region["masked_mse"]:.6f}.
- The model reduced affected-region error by about {affected_mse_ratio:.1f}x compared with the identity baseline.

Interpretation:
- The model learns useful contaminant removal on held-out synthetic blends.
- The strongest results occur when the contaminant is visually separable.
- The weakest results occur when the contaminant overlaps the target core or when the scene is genuinely ambiguous.

Important note:
- Original generation easy/medium/hard labels are crude.
- Blend severity bins reflect image-level contaminant damage, not model difficulty.
- Model difficulty/failure should be read from model affected error and model-improvement ratios.
""".strip()
log_path.write_text(log_text + "\n", encoding="utf-8")

saved_outputs = {
    "model_checkpoint": checkpoint_path.relative_to(PROJECT_ROOT),
    "figures": sorted(path.name for path in figure_dir.glob("direct_unet_*_example.png")),
    "log": log_path.relative_to(PROJECT_ROOT),
}
saved_outputs


## 16. Next Steps

- Run a balanced hard-case stress test with more severe overlaps.
- Increase training data and evaluation size for final tables.
- Test a residual-prediction variant that predicts the contaminant layer and subtracts it from the blend.
- Prepare final report figures and paper tables after the full experiment set is complete.
